In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')

In [2]:
from openai import OpenAI

client = OpenAI()

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="test"
)


In [3]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

try:
    indexes = pc.list_indexes()
    print(f"✓ Pinecone connected - Existing indexes: {[idx.name for idx in indexes]}")
except Exception as e:
    print(f"✗ Pinecone connection failed: {e}")

✓ Pinecone connected - Existing indexes: ['1111', 'goty2023', 'lol-rag']


In [4]:
INDEX_NAME = "lol-rag"
DIMENSION = 1536

# Check if index exists
existing_indexes = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    print(f"Creating Pinecone index: {INDEX_NAME}")
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"  
        )
    )
    print(f"✓ Index '{INDEX_NAME}' created")
else:
    print(f"✓ Index '{INDEX_NAME}' already exists")

✓ Index 'lol-rag' already exists


In [5]:
# Connect to the index
index = pc.Index(INDEX_NAME)

/Users/macbookpro/Desktop/riot_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Test fetching one JSON file from CommunityDragon
import requests

BASE_URL = "https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/"

champions = []

FILES = {
    "summary": "champion-summary.json",
    "items": "items.json",
    "perks": "perks.json",
    "spells": "summoner-spells.json"
}

try:
    response = requests.get(f"{BASE_URL}champion-summary.json")
    response.raise_for_status()
    champion_summary = response.json()
    
    print(f"✓ Successfully fetched champion-summary.json")
    print(f"  - Number of champions: {len(champion_summary)}")
    print(f"  - Sample champion: {champion_summary[0]['name'] if champion_summary else 'N/A'}")
except Exception as e:
    print(f"✗ Failed to fetch data: {e}")

✓ Successfully fetched champion-summary.json
  - Number of champions: 192
  - Sample champion: None


In [7]:
import requests
import time

# 1. Fetch the manifest
summary_url = f"{BASE_URL}champion-summary.json"
summary_data = requests.get(summary_url).json()

# 2. Filter IDs: Must be > 0 and < 1000
# This removes 'None' (-1) and TFT/Special units (> 1000)
valid_entries = [c for c in summary_data if 0 < c['id'] < 1000]

print(f"Found {len(valid_entries)} standard champions to fetch.")

champion_full_data = []

# 3. Loop and populate the list
for entry in valid_entries:
    champ_id = entry['id']
    name = entry['name']
    
    try:
        detail_url = f"{BASE_URL}champions/{champ_id}.json"
        response = requests.get(detail_url)
        response.raise_for_status()
        
        # Add to our list
        champion_full_data.append(response.json())
        
        # Optional: Print progress every 20 champs so you know it's working
        if len(champion_full_data) % 20 == 0:
            print(f"✓ Fetched {len(champion_full_data)} champions...")
            
    except Exception as e:
        print(f"⚠ Failed to fetch {name} (ID: {champ_id}): {e}")

print(f"\nSuccess! 'champion_full_data' now contains {len(champion_full_data)} detailed objects.")

Found 172 standard champions to fetch.
✓ Fetched 20 champions...
✓ Fetched 40 champions...
✓ Fetched 60 champions...
✓ Fetched 80 champions...
✓ Fetched 100 champions...
✓ Fetched 120 champions...
✓ Fetched 140 champions...
✓ Fetched 160 champions...

Success! 'champion_full_data' now contains 172 detailed objects.


In [8]:
%store champion_full_data

Stored 'champion_full_data' (list)


In [9]:
from langchain_community.graphs import Neo4jGraph 

graph = Neo4jGraph(
    url="neo4j+s://f5886a71.databases.neo4j.io",
    username="neo4j",
    password="fFCqS5-Ak3ZLppY0p1e2rrL1ZN_FJwt7oH_McxlCh5o"
)

/var/folders/_n/46drt9kd1_76dqk_vj0xk9qm0000gn/T/ipykernel_1869/4030586405.py:3: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  graph = Neo4jGraph(


In [10]:
# Query 1: Get all champions with their metadata
champions_query = """
MATCH (c:Champion)
OPTIONAL MATCH (c)-[:HAS_ROLE]->(r:Role)
RETURN c.id as champion_id,
       c.name as champion_name,
       c.title as title,
       collect(DISTINCT r.name) as roles
ORDER BY c.name
"""

champions_data = graph.query(champions_query)
print(f"✓ Loaded {len(champions_data)} champions from Neo4j")
for champ in champions_data[:3]:
    print(f"  - {champ['champion_name']}: {', '.join(champ['roles'])}")

# Query 2: Get all abilities with their metadata
abilities_query = """
MATCH (c:Champion)-[:HAS_ABILITY]->(a:Ability)
OPTIONAL MATCH (a)-[:DEALS]->(dt:DamageType)
OPTIONAL MATCH (a)-[:APPLIES]->(cc:CrowdControl)
OPTIONAL MATCH (a)-[:SCALES_WITH]->(s:Stat)
RETURN a.id as ability_id,
       a.name as ability_name,
       a.slot as slot,
       a.description as description,
       c.id as champion_id,
       c.name as champion_name,
       collect(DISTINCT dt.name) as damage_types,
       collect(DISTINCT cc.name) as cc_types,
       collect(DISTINCT s.name) as scaling_stats
ORDER BY c.name, a.slot
"""

abilities_data = graph.query(abilities_query)
print(f"\n✓ Loaded {len(abilities_data)} abilities from Neo4j")
for ability in abilities_data[:3]:
    print(f"  - {ability['champion_name']} [{ability['slot']}] {ability['ability_name']}")

✓ Loaded 172 champions from Neo4j
  - Aatrox: Fighter
  - Ahri: Mage, Assassin
  - Akali: Assassin

✓ Loaded 860 abilities from Neo4j
  - Aatrox [E] Umbral Dash
  - Aatrox [Passive] Deathbringer Stance
  - Aatrox [Q] The Darkin Blade


In [11]:
def create_embedding(text):
    """
    Create embedding using OpenAI API.
    
    Args:
        text: String to embed
    
    Returns:
        List of floats (1536 dimensions)
    """
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# Test the function
test_text = "Annie is a fire mage"
test_embedding = create_embedding(test_text)

print(f"✓ Embedding function works")
print(f"  Text: '{test_text}'")
print(f"  Embedding length: {len(test_embedding)}")
print(f"  First 5 values: {test_embedding[:5]}")

✓ Embedding function works
  Text: 'Annie is a fire mage'
  Embedding length: 1536
  First 5 values: [0.05034066364169121, -0.01893211342394352, -0.02710023894906044, -0.04343648999929428, -0.021106654778122902]


In [12]:
def prepare_ability_vector(ability_data):
    """
    Prepare an ability for Pinecone upload.
    """
    # Create rich text for embedding
    text_parts = [
        f"Champion: {ability_data['champion_name']}",
        f"Ability: {ability_data['ability_name']} ({ability_data['slot']})",
        f"Description: {ability_data['description']}"
    ]
    
    # Add context from relationships
    if ability_data['damage_types']:
        damage = [d for d in ability_data['damage_types'] if d]  # Filter empty
        if damage:
            text_parts.append(f"Damage: {', '.join(damage)}")
    
    if ability_data['cc_types']:
        cc = [c for c in ability_data['cc_types'] if c]
        if cc:
            text_parts.append(f"Crowd Control: {', '.join(cc)}")
    
    if ability_data['scaling_stats']:
        stats = [s for s in ability_data['scaling_stats'] if s]
        if stats:
            text_parts.append(f"Scales with: {', '.join(stats)}")
    
    text = ". ".join(text_parts)
    
    # Create embedding
    embedding = create_embedding(text)
    
    # Prepare vector
    return {
        'id': ability_data['ability_id'],
        'values': embedding,
        'metadata': {
            'type': 'ability',
            'champion_id': ability_data['champion_id'],
            'champion_name': ability_data['champion_name'],
            'ability_name': ability_data['ability_name'],
            'slot': ability_data['slot'],
            'text': text[:1000]  # Store truncated text for display
        }
    }

print("✓ Ability vector preparation function ready")

✓ Ability vector preparation function ready


In [16]:
def prepare_champion_vector(champion_data):
    """
    Prepare a champion profile for Pinecone upload.
    """
    # Create text representation
    text_parts = [
        f"{champion_data['champion_name']} - {champion_data['title']}",
        f"Roles: {', '.join(champion_data['roles'])}"
    ]
    
    # Note: We'll add champion bio later from the JSON
    # For now, this is enough for testing
    
    text = ". ".join(text_parts)
    
    # Create embedding
    embedding = create_embedding(text)
    
    # Prepare vector
    return {
        'id': f"champion_{champion_data['champion_id']}",
        'values': embedding,
        'metadata': {
            'type': 'champion',
            'champion_id': champion_data['champion_id'],
            'champion_name': champion_data['champion_name'],
            'title': champion_data['title'],
            'roles': ','.join(champion_data['roles']),  # Pinecone doesn't support lists
            'text': text
        }
    }

print("✓ Champion vector preparation function ready")

✓ Champion vector preparation function ready


In [13]:
print("Creating ability embeddings...\n")

ability_vectors = []

for i, ability in enumerate(abilities_data, 1):
    try:
        vector = prepare_ability_vector(ability)
        ability_vectors.append(vector)
        
        if i % 10 == 0:
            print(f"  ✓ Processed {i}/{len(abilities_data)} abilities")
    except Exception as e:
        print(f"  ⚠️ Error on {ability['champion_name']} - {ability['ability_name']}: {e}")

print(f"\n✓ Created {len(ability_vectors)} ability embeddings")

Creating ability embeddings...

  ✓ Processed 10/860 abilities
  ✓ Processed 20/860 abilities
  ✓ Processed 30/860 abilities
  ✓ Processed 40/860 abilities
  ✓ Processed 50/860 abilities
  ✓ Processed 60/860 abilities
  ✓ Processed 70/860 abilities
  ✓ Processed 80/860 abilities
  ✓ Processed 90/860 abilities
  ✓ Processed 100/860 abilities
  ✓ Processed 110/860 abilities
  ✓ Processed 120/860 abilities
  ✓ Processed 130/860 abilities
  ✓ Processed 140/860 abilities
  ✓ Processed 150/860 abilities
  ✓ Processed 160/860 abilities
  ✓ Processed 170/860 abilities
  ✓ Processed 180/860 abilities
  ✓ Processed 190/860 abilities
  ✓ Processed 200/860 abilities
  ✓ Processed 210/860 abilities
  ✓ Processed 220/860 abilities
  ✓ Processed 230/860 abilities
  ✓ Processed 240/860 abilities
  ✓ Processed 250/860 abilities
  ✓ Processed 260/860 abilities
  ✓ Processed 270/860 abilities
  ✓ Processed 280/860 abilities
  ✓ Processed 290/860 abilities
  ✓ Processed 300/860 abilities
  ✓ Processed 310

In [17]:
print("Creating champion embeddings...\n")

champion_vectors = []

for champion in champions_data:
    try:
        vector = prepare_champion_vector(champion)
        champion_vectors.append(vector)
        print(f"  ✓ {champion['champion_name']}")
    except Exception as e:
        print(f"  ⚠️ Error on {champion['champion_name']}: {e}")

print(f"\n✓ Created {len(champion_vectors)} champion embeddings")

Creating champion embeddings...

  ✓ Aatrox
  ✓ Ahri
  ✓ Akali
  ✓ Akshan
  ✓ Alistar
  ✓ Ambessa
  ✓ Amumu
  ✓ Anivia
  ✓ Annie
  ✓ Aphelios
  ✓ Ashe
  ✓ Aurelion Sol
  ✓ Aurora
  ✓ Azir
  ✓ Bard
  ✓ Bel'Veth
  ✓ Blitzcrank
  ✓ Brand
  ✓ Braum
  ✓ Briar
  ✓ Caitlyn
  ✓ Camille
  ✓ Cassiopeia
  ✓ Cho'Gath
  ✓ Corki
  ✓ Darius
  ✓ Diana
  ✓ Dr. Mundo
  ✓ Draven
  ✓ Ekko
  ✓ Elise
  ✓ Evelynn
  ✓ Ezreal
  ✓ Fiddlesticks
  ✓ Fiora
  ✓ Fizz
  ✓ Galio
  ✓ Gangplank
  ✓ Garen
  ✓ Gnar
  ✓ Gragas
  ✓ Graves
  ✓ Gwen
  ✓ Hecarim
  ✓ Heimerdinger
  ✓ Hwei
  ✓ Illaoi
  ✓ Irelia
  ✓ Ivern
  ✓ Janna
  ✓ Jarvan IV
  ✓ Jax
  ✓ Jayce
  ✓ Jhin
  ✓ Jinx
  ✓ K'Sante
  ✓ Kai'Sa
  ✓ Kalista
  ✓ Karma
  ✓ Karthus
  ✓ Kassadin
  ✓ Katarina
  ✓ Kayle
  ✓ Kayn
  ✓ Kennen
  ✓ Kha'Zix
  ✓ Kindred
  ✓ Kled
  ✓ Kog'Maw
  ✓ LeBlanc
  ✓ Lee Sin
  ✓ Leona
  ✓ Lillia
  ✓ Lissandra
  ✓ Lucian
  ✓ Lulu
  ✓ Lux
  ✓ Malphite
  ✓ Malzahar
  ✓ Maokai
  ✓ Master Yi
  ✓ Mel
  ✓ Milio
  ✓ Miss Fortune
  ✓ Mordekaiser
  ✓ Morg

In [18]:
# Combine all vectors
all_vectors = ability_vectors + champion_vectors

print(f"Uploading {len(all_vectors)} vectors to Pinecone...")

# Upload in batches
batch_size = 100
for i in range(0, len(all_vectors), batch_size):
    batch = all_vectors[i:i+batch_size]
    index.upsert(vectors=batch)
    print(f"  ✓ Uploaded batch {i//batch_size + 1}")

print(f"\n🎉 Upload complete!")

# Verify
time.sleep(2)  # Wait for index to update
stats = index.describe_index_stats()
print(f"\nIndex stats:")
print(f"  Total vectors: {stats['total_vector_count']}")

Uploading 1032 vectors to Pinecone...
  ✓ Uploaded batch 1
  ✓ Uploaded batch 2
  ✓ Uploaded batch 3
  ✓ Uploaded batch 4
  ✓ Uploaded batch 5
  ✓ Uploaded batch 6
  ✓ Uploaded batch 7
  ✓ Uploaded batch 8
  ✓ Uploaded batch 9
  ✓ Uploaded batch 10
  ✓ Uploaded batch 11

🎉 Upload complete!

Index stats:
  Total vectors: 1032


In [19]:
def semantic_search(query, top_k=5, filter_type=None):
    """
    Search Pinecone using semantic similarity.
    
    Args:
        query: Natural language query
        top_k: Number of results to return
        filter_type: Optional filter ('ability' or 'champion')
    
    Returns:
        List of matching results with scores
    """
    # Create embedding for query
    query_embedding = create_embedding(query)
    
    # Build filter if specified
    query_filter = None
    if filter_type:
        query_filter = {'type': filter_type}
    
    # Search
    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        filter=query_filter
    )
    
    return results['matches']

print("✓ Search function ready")

✓ Search function ready


In [20]:
# Test 1: Find fire-related abilities
print("🔥 Query: 'fire abilities'\n")

results = semantic_search("fire abilities", top_k=5, filter_type='ability')

print("Top matches:")
for i, match in enumerate(results, 1):
    metadata = match['metadata']
    score = match['score']
    print(f"{i}. {metadata['champion_name']} - {metadata['ability_name']} [{metadata['slot']}]")
    print(f"   Similarity: {score:.3f}")
    print(f"   Preview: {metadata['text'][:100]}...\n")

🔥 Query: 'fire abilities'

Top matches:
1. Zeri - Burst Fire [Q]
   Similarity: 0.477
   Preview: Champion: Zeri. Ability: Burst Fire (Q). Description: Burst Fire shoots a burst of 7 rounds that dea...

2. Smolder - MMOOOMMMM! [R]
   Similarity: 0.450
   Preview: Champion: Smolder. Ability: MMOOOMMMM! (R). Description: Smolder calls his mom to breath fire from a...

3. Smolder - Super Scorcher Breath [Q]
   Similarity: 0.448
   Preview: Champion: Smolder. Ability: Super Scorcher Breath (Q). Description: Smolder breathes fire on an enem...

4. Brand - Pyroclasm [R]
   Similarity: 0.445
   Preview: Champion: Brand. Ability: Pyroclasm (R). Description: Brand unleashes a devastating torrent of fire ...

5. Brand - Conflagration [E]
   Similarity: 0.439
   Preview: Champion: Brand. Ability: Conflagration (E). Description: Brand conjures a powerful blast at his tar...



In [21]:
# Test 2: Find stun abilities
print("⚡ Query: 'stun abilities'\n")

results = semantic_search("stun abilities", top_k=5, filter_type='ability')

print("Top matches:")
for i, match in enumerate(results, 1):
    metadata = match['metadata']
    score = match['score']
    print(f"{i}. {metadata['champion_name']} - {metadata['ability_name']} [{metadata['slot']}]")
    print(f"   Similarity: {score:.3f}\n")

⚡ Query: 'stun abilities'

Top matches:
1. Aurora - Between Worlds [R]
   Similarity: 0.536

2. Udyr - Blazing Stampede [E]
   Similarity: 0.532

3. Kennen - Mark of the Storm [Passive]
   Similarity: 0.524

4. Rell - Shattering Strike [Q]
   Similarity: 0.522

5. Taric - Dazzle [E]
   Similarity: 0.520



In [22]:
# Test 3: Find champions by concept
print("👧 Query: 'child mage'\n")

results = semantic_search("child mage", top_k=3, filter_type='champion')

print("Top matches:")
for i, match in enumerate(results, 1):
    metadata = match['metadata']
    score = match['score']
    print(f"{i}. {metadata['champion_name']} - {metadata['title']}")
    print(f"   Roles: {metadata['roles']}")
    print(f"   Similarity: {score:.3f}\n")

👧 Query: 'child mage'

Top matches:
1. Annie - the Dark Child
   Roles: Mage,Support
   Similarity: 0.490

2. Kennen - the Heart of the Tempest
   Roles: Mage
   Similarity: 0.457

3. Smolder - the Fiery Fledgling
   Roles: Mage,Marksman
   Similarity: 0.453



In [23]:
# Test 4: Find tanky champions
print("🛡️ Query: 'tanky defensive champions'\n")

results = semantic_search("tanky defensive champions", top_k=5, filter_type='champion')

print("Top matches:")
for i, match in enumerate(results, 1):
    metadata = match['metadata']
    score = match['score']
    print(f"{i}. {metadata['champion_name']} ({metadata['roles']})")
    print(f"   Similarity: {score:.3f}\n")

🛡️ Query: 'tanky defensive champions'

Top matches:
1. Taric (Support,Tank)
   Similarity: 0.472

2. Tahm Kench (Support,Tank)
   Similarity: 0.445

3. Garen (Fighter,Tank)
   Similarity: 0.402

4. Sejuani (Tank)
   Similarity: 0.400

5. Nautilus (Support,Tank)
   Similarity: 0.394



In [24]:
# Test 5: Find protective abilities
print("🛡️ Query: 'defensive protective shield abilities'\n")

results = semantic_search("defensive protective shield abilities", top_k=5, filter_type='ability')

print("Top matches:")
for i, match in enumerate(results, 1):
    metadata = match['metadata']
    score = match['score']
    print(f"{i}. {metadata['champion_name']} - {metadata['ability_name']}")
    print(f"   Similarity: {score:.3f}\n")

🛡️ Query: 'defensive protective shield abilities'

Top matches:
1. Shen - Ki Barrier
   Similarity: 0.512

2. Vex - Personal Space
   Similarity: 0.493

3. Mel - Rebuttal
   Similarity: 0.491

4. Morgana - Black Shield
   Similarity: 0.486

5. Camille - Adaptive Defenses
   Similarity: 0.486



In [29]:
def hybrid_search(semantic_query, neo4j_filters=None, top_k=10):
    """
    Combine semantic search (Pinecone) with structured filters (Neo4j).
    
    Args:
        semantic_query: Natural language query for Pinecone
        neo4j_filters: Dict of Neo4j constraints
            Examples:
            - {'role': 'Mage'}
            - {'attack_type': 'Ranged'}
            - {'role': 'Mage', 'attack_type': 'Ranged'}
        top_k: Number of semantic results to retrieve
    
    Returns:
        Combined results
    """
    # Step 1: Get semantic results from Pinecone
    pinecone_results = semantic_search(semantic_query, top_k=top_k)
    
    # Extract champion IDs from results
    champion_ids = set()
    for match in pinecone_results:
        champion_ids.add(match['metadata']['champion_id'])
    
    # Step 2: Apply Neo4j filters
    if neo4j_filters:
        # Build Cypher query
        where_clauses = []
        params = {'champion_ids': list(champion_ids)}
        
        base_query = "MATCH (c:Champion) WHERE c.id IN $champion_ids"
        
        if 'role' in neo4j_filters:
            base_query += " MATCH (c)-[:HAS_ROLE]->(r:Role {name: $role})"
            params['role'] = neo4j_filters['role']
        
        if 'attack_type' in neo4j_filters:
            base_query += " MATCH (c)-[:ATTACKS_WITH]->(at:AttackType {name: $attack_type})"
            params['attack_type'] = neo4j_filters['attack_type']
        
        base_query += " RETURN DISTINCT c.id as champion_id, c.name as champion_name"
        
        filtered_champions = graph.query(base_query, params=params)
        filtered_ids = {c['champion_id'] for c in filtered_champions}
    else:
        filtered_ids = champion_ids
    
    # Step 3: Filter Pinecone results by Neo4j criteria
    final_results = []
    for match in pinecone_results:
        if match['metadata']['champion_id'] in filtered_ids:
            final_results.append(match)
    
    return final_results

print("✓ Hybrid search function ready")

✓ Hybrid search function ready


In [41]:
# Test hybrid search: "Fire abilities from ranged mages"
results = hybrid_search(
    semantic_query="spell immune",
    neo4j_filters={
        'role': 'Marksman',
        'attack_type': 'Ranged'
    },
    top_k=10
)

print(f"Found {len(results)} matches:\n")
for i, match in enumerate(results, 1):
    metadata = match['metadata']
    if metadata['type'] == 'ability':
        print(f"{i}. {metadata['champion_name']} - {metadata['ability_name']} [{metadata['slot']}]")
        print(f"   Similarity: {match['score']:.3f}\n")

Found 2 matches:

1. Sivir - Spell Shield [E]
   Similarity: 0.438

2. Senna - Curse of the Black Mist [E]
   Similarity: 0.406

